# Pratica — Lezione 3. Processi decisionali di Markov (MDP)

Questa pratica è tutta tua. Lavorerai in autonomia con lo stesso `GridWorld` che hai costruito in classe e risolverai due esercizi sul **Processo Decisionale di Markov** (MDP) e sull'**equazione di Bellman dell'aspettativa**.

**Come lavorare con questo notebook:**
1. Esegui prima la sezione "Preparazione", che ricostruisce l'ambiente della lezione
2. In ogni esercizio, completa le righe contrassegnate con `# TODO`
3. Esegui la tua soluzione e confronta il risultato con quanto descritto nella consegna
4. Solo dopo, apri la soluzione nascosta per verificare la tua risposta

> **Non guardare la soluzione prima di averci provato.** Il valore di questo esercizio sta nel confrontarsi un po' con l'equazione di Bellman, non nel leggere la risposta.

## Preparazione: recupera il GridWorld della lezione

Queste celle riproducono esattamente la classe `GridWorld` e le funzioni `politica_aleatoria`, `evaluacion_iterativa_politica` e `imprimir_valores` che hai costruito nel notebook teorico. Gli esercizi di oggi riutilizzano questi stessi nomi, quindi non cambiarli.

In [ ]:
import numpy as np

ACCIONES = ["arriba", "abajo", "izquierda", "derecha"]

DESPLAZAMIENTOS = {
    "arriba": (-1, 0),
    "abajo": (1, 0),
    "izquierda": (0, -1),
    "derecha": (0, 1),
}

In [ ]:
class GridWorld:
    """
    Un mundo-cuadrícula de n_filas x n_columnas con una casilla terminal (la meta).

    - Los estados son tuplas (fila, columna).
    - Las acciones son las cuatro direcciones cardinales.
    - Las transiciones son deterministas: moverse contra un muro o el
      borde del tablero deja al agente en la misma casilla.
    - La recompensa es 'recompensa_paso' en cada movimiento (por defecto -1),
      salvo en la casilla terminal, donde el episodio ya ha terminado.
    """

    def __init__(self, n_filas=4, n_columnas=4, estados_terminales=None, recompensa_paso=-1.0):
        self.n_filas = n_filas
        self.n_columnas = n_columnas
        self.recompensa_paso = recompensa_paso

        # Por defecto, la meta es la esquina inferior derecha
        if estados_terminales is None:
            estados_terminales = [(n_filas - 1, n_columnas - 1)]
        self.estados_terminales = set(estados_terminales)

        # Lista de todos los estados (fila, columna)
        self.estados = [(f, c) for f in range(n_filas) for c in range(n_columnas)]
        self.acciones = ACCIONES

    def es_terminal(self, estado):
        return estado in self.estados_terminales

    def estado_siguiente(self, estado, accion):
        """Devuelve la casilla resultante de aplicar 'accion' desde 'estado'.
        Si el movimiento choca con el borde del tablero, el agente se queda quieto."""
        if self.es_terminal(estado):
            return estado

        df, dc = DESPLAZAMIENTOS[accion]
        fila, columna = estado
        nueva_fila = max(0, min(self.n_filas - 1, fila + df))
        nueva_columna = max(0, min(self.n_columnas - 1, columna + dc))
        return (nueva_fila, nueva_columna)

    def recompensa(self, estado, accion, estado_siguiente):
        """Recompensa inmediata de la transición. En este GridWorld es fija
        (-1 por paso), pero se recibe (s, a, s') completo para poder
        ampliar el ejemplo más adelante (por ejemplo, recompensas distintas por casilla)."""
        if self.es_terminal(estado):
            return 0.0
        return self.recompensa_paso


def politica_aleatoria(mundo):
    """Construye una política estocástica que, en cada estado, elige
    cualquiera de las cuatro acciones con la misma probabilidad (0.25).
    Se representa como un diccionario: politica[estado][accion] = probabilidad."""
    probabilidad_uniforme = 1.0 / len(mundo.acciones)
    return {
        estado: {accion: probabilidad_uniforme for accion in mundo.acciones}
        for estado in mundo.estados
    }


def evaluacion_iterativa_politica(mundo, politica, gamma=0.9, umbral=1e-4, max_iteraciones=1000):
    """
    Calcula Vpi(s) para una política fija aplicando repetidamente
    la ecuación de Bellman de expectación:

        Vpi(s) = suma_a pi(a|s) * suma_s' P(s'|s,a) * [R(s,a,s') + gamma * Vpi(s')]

    Devuelve V como un diccionario {estado: valor} y el número de barridos usados.
    """
    V = {estado: 0.0 for estado in mundo.estados}

    for iteracion in range(1, max_iteraciones + 1):
        delta_maximo = 0.0
        V_nuevo = dict(V)  # barrido síncrono: usamos siempre los valores del barrido anterior

        for estado in mundo.estados:
            if mundo.es_terminal(estado):
                V_nuevo[estado] = 0.0
                continue

            valor_esperado = 0.0
            for accion in mundo.acciones:
                prob_accion = politica[estado][accion]
                estado_sig = mundo.estado_siguiente(estado, accion)
                r = mundo.recompensa(estado, accion, estado_sig)
                valor_esperado += prob_accion * (r + gamma * V[estado_sig])

            V_nuevo[estado] = valor_esperado
            delta_maximo = max(delta_maximo, abs(V_nuevo[estado] - V[estado]))

        V = V_nuevo
        if delta_maximo < umbral:
            break

    return V, iteracion


def imprimir_valores(mundo, V):
    """Muestra V como una cuadrícula legible, para inspeccionar el resultado."""
    for f in range(mundo.n_filas):
        fila_textos = []
        for c in range(mundo.n_columnas):
            fila_textos.append(f"{V[(f, c)]:6.2f}")
        print(" ".join(fila_textos))

### Linea di base: il GridWorld originale

Esegui la cella seguente per avere a portata di mano lo stesso riferimento visto in classe: il GridWorld 4x4 con l'obiettivo nell'angolo in basso a destra, la sua politica casuale e la sua Vπ già convergente. Userai `mundo`, `politica` e `V` nell'esercizio 2, quindi non sovrascriverli.

In [ ]:
mundo = GridWorld(n_filas=4, n_columnas=4, recompensa_paso=-1.0)
politica = politica_aleatoria(mundo)
V, num_barridos = evaluacion_iterativa_politica(mundo, politica, gamma=0.9, umbral=1e-4)

print(f"Convergió en {num_barridos} barridos.\n")
imprimir_valores(mundo, V)

---

## Esercizio 1: cambia la disposizione delle ricompense e completa la valutazione

Il `GridWorld` del capitolo mette sempre l'obiettivo nell'angolo in basso a destra, ma la classe accetta qualsiasi lista di `estados_terminales`. In questo esercizio dovrai:

1. Costruire un nuovo `GridWorld` con l'obiettivo in un'altra posizione —per esempio al centro della griglia, `[(1, 1)]`— oppure con **due** obiettivi contemporaneamente —per esempio `[(0, 3), (3, 0)]`—
2. Completare gli spazi vuoti del ciclo di valutazione iterativa della politica per calcolare Vπ su questo nuovo mondo

**Suggerimenti:**
- Il costruttore è `GridWorld(n_filas=4, n_columnas=4, estados_terminales=..., recompensa_paso=-1.0)`
- I tre spazi da riempire all'interno del ciclo corrispondono, in ordine, a: lo stato successivo (`mundo.estado_siguiente`), la ricompensa immediata (`mundo.recompensa`) e il termine dell'equazione di Bellman dell'aspettativa: `prob_accion * (r + gamma * V[estado_sig])`
- Se sposti l'obiettivo al centro, aspettati di vedere i valori più alti (meno negativi) attorno a `(1, 1)` invece che nell'angolo
- Se usi due obiettivi, ogni casella "vede" il percorso più breve verso uno dei due

In [ ]:
# Paso 1: elige dónde van las metas de tu nuevo GridWorld
# Ejemplos: [(1, 1)]              -> una única meta en el centro
#           [(0, 3), (3, 0)]      -> dos metas en esquinas opuestas
nuevos_estados_terminales = None  # TODO: sustituye None por una lista de tuplas (fila, columna)

mundo_ejercicio = GridWorld(
    n_filas=4,
    n_columnas=4,
    estados_terminales=nuevos_estados_terminales,
    recompensa_paso=-1.0,
)

politica_ejercicio = politica_aleatoria(mundo_ejercicio)

In [ ]:
def evaluacion_iterativa_politica_ejercicio(mundo, politica, gamma=0.9, umbral=1e-4, max_iteraciones=1000):
    """Misma evaluación iterativa de políticas que en clase, pero con
    tres huecos por completar dentro del bucle sobre las acciones."""
    V = {estado: 0.0 for estado in mundo.estados}

    for iteracion in range(1, max_iteraciones + 1):
        delta_maximo = 0.0
        V_nuevo = dict(V)

        for estado in mundo.estados:
            if mundo.es_terminal(estado):
                V_nuevo[estado] = 0.0
                continue

            valor_esperado = 0.0
            for accion in mundo.acciones:
                prob_accion = politica[estado][accion]

                # TODO: calcula el estado siguiente de aplicar 'accion' desde 'estado'
                estado_sig = None

                # TODO: calcula la recompensa inmediata de esa transición (estado, accion, estado_sig)
                r = None

                # TODO: acumula en valor_esperado el sumando de Bellman de expectación:
                #       prob_accion * (r + gamma * V[estado_sig])
                valor_esperado += 0

            V_nuevo[estado] = valor_esperado
            delta_maximo = max(delta_maximo, abs(V_nuevo[estado] - V[estado]))

        V = V_nuevo
        if delta_maximo < umbral:
            break

    return V, iteracion


V_ejercicio, barridos_ejercicio = evaluacion_iterativa_politica_ejercicio(
    mundo_ejercicio, politica_ejercicio, gamma=0.9
)

print(f"Convergió en {barridos_ejercicio} barridos.\n")
imprimir_valores(mundo_ejercicio, V_ejercicio)

<details>
<summary>💡 Vedi la soluzione</summary>

```python
# Passo 1: scegliamo l'obiettivo al centro della griglia
nuevos_estados_terminales = [(1, 1)]

mundo_ejercicio = GridWorld(
    n_filas=4,
    n_columnas=4,
    estados_terminales=nuevos_estados_terminales,
    recompensa_paso=-1.0,
)

politica_ejercicio = politica_aleatoria(mundo_ejercicio)


def evaluacion_iterativa_politica_ejercicio(mundo, politica, gamma=0.9, umbral=1e-4, max_iteraciones=1000):
    V = {estado: 0.0 for estado in mundo.estados}

    for iteracion in range(1, max_iteraciones + 1):
        delta_maximo = 0.0
        V_nuevo = dict(V)

        for estado in mundo.estados:
            if mundo.es_terminal(estado):
                V_nuevo[estado] = 0.0
                continue

            valor_esperado = 0.0
            for accion in mundo.acciones:
                prob_accion = politica[estado][accion]

                # Stato raggiunto dopo aver applicato l'azione
                estado_sig = mundo.estado_siguiente(estado, accion)

                # Ricompensa immediata di quella transizione
                r = mundo.recompensa(estado, accion, estado_sig)

                # Termine dell'equazione di Bellman dell'aspettativa:
                # pi(a|s) * [R(s,a,s') + gamma * Vpi(s')]
                valor_esperado += prob_accion * (r + gamma * V[estado_sig])

            V_nuevo[estado] = valor_esperado
            delta_maximo = max(delta_maximo, abs(V_nuevo[estado] - V[estado]))

        V = V_nuevo
        if delta_maximo < umbral:
            break

    return V, iteracion


V_ejercicio, barridos_ejercicio = evaluacion_iterativa_politica_ejercicio(
    mundo_ejercicio, politica_ejercicio, gamma=0.9
)

print(f"Convergenza in {barridos_ejercicio} passate.\n")
imprimir_valores(mundo_ejercicio, V_ejercicio)

# Output atteso approssimativo (l'obiettivo ora è al centro, (1, 1)):
#  -5.82  -4.89  -6.59  -7.57
#  -4.89   0.00  -5.78  -7.48
#  -6.59  -5.78  -7.19  -7.97
#  -7.57  -7.48  -7.97  -8.34
#
# Fai attenzione al pattern: il valore meno negativo (0.00) non è più nell'angolo,
# ma in (1, 1), e le altre caselle si riorganizzano attorno al nuovo obiettivo
# esattamente come prima si riorganizzavano attorno all'angolo in basso a destra.

# Variante con due obiettivi, per verificare che il pattern si ripete:
mundo_dos_metas = GridWorld(
    n_filas=4, n_columnas=4,
    estados_terminales=[(0, 3), (3, 0)],
    recompensa_paso=-1.0,
)
politica_dos_metas = politica_aleatoria(mundo_dos_metas)
V_dos_metas, barridos_dos_metas = evaluacion_iterativa_politica_ejercicio(
    mundo_dos_metas, politica_dos_metas, gamma=0.9
)
print(f"\nDue obiettivi — convergenza in {barridos_dos_metas} passate.\n")
imprimir_valores(mundo_dos_metas, V_dos_metas)
```
</details>

---

## Esercizio 2: l'equazione di Bellman dell'aspettativa, a mano

In questo esercizio non userai `evaluacion_iterativa_politica`. Calcolerai Vπ per **un singolo stato** applicando direttamente l'equazione di Bellman dell'aspettativa, sommando termine per termine sulle quattro azioni:

```
Vπ(s) = Σ_a π(a|s) · [R(s,a,s') + γ·Vπ(s')]
```

Usa il `mundo`, la `politica` e la `V` che hai già calcolato nella sezione "Linea di base" qui sopra: per ogni azione ti serve `V[estado_sig]`, cioè il valore che lo stato vicino ha già secondo la valutazione completa. Alla fine, confronta il tuo risultato con `V[estado_elegido]`: dovrebbero coincidere, perché in uno stato convergente l'equazione di Bellman è già soddisfatta.

**Suggerimenti:**
- Percorri `mundo.acciones` con un ciclo for
- Per ogni azione ti servono: la probabilità `politica[estado_elegido][accion]`, lo stato successivo `mundo.estado_siguiente(estado_elegido, accion)`, la ricompensa `mundo.recompensa(...)` e il valore `V[estado_sig]`
- Poiché `evaluacion_iterativa_politica` si ferma appena `delta_maximo < umbral`, i due risultati saranno praticamente uguali, ma non identici bit per bit: la differenza dovrebbe essere minore della soglia `umbral` (1e-4) usata per calcolare `V`

In [ ]:
estado_elegido = (2, 3)  # TODO: puedes probar con cualquier estado no terminal del mundo
gamma = 0.9

valor_manual = 0.0

for accion in mundo.acciones:
    # TODO: obtén la probabilidad de esta acción en politica[estado_elegido]
    prob_accion = None

    # TODO: calcula el estado siguiente con mundo.estado_siguiente(...)
    estado_sig = None

    # TODO: calcula la recompensa inmediata con mundo.recompensa(...)
    r = None

    # TODO: acumula en valor_manual el sumando prob_accion * (r + gamma * V[estado_sig])
    valor_manual += 0

print(f"Valor calculado a mano para {estado_elegido}: {valor_manual:.4f}")
print(f"Valor de la función genérica para {estado_elegido}: {V[estado_elegido]:.4f}")
print(f"Diferencia: {abs(valor_manual - V[estado_elegido]):.6f}")

<details>
<summary>💡 Vedi la soluzione</summary>

```python
estado_elegido = (2, 3)
gamma = 0.9

valor_manual = 0.0

for accion in mundo.acciones:
    # Probabilità che la politica assegna a questa azione nello stato scelto
    prob_accion = politica[estado_elegido][accion]

    # Stato raggiunto dopo aver applicato l'azione
    estado_sig = mundo.estado_siguiente(estado_elegido, accion)

    # Ricompensa immediata di quella transizione
    r = mundo.recompensa(estado_elegido, accion, estado_sig)

    # Termine dell'equazione di Bellman dell'aspettativa
    valor_manual += prob_accion * (r + gamma * V[estado_sig])

print(f"Valore calcolato a mano per {estado_elegido}: {valor_manual:.4f}")
print(f"Valore della funzione generica per {estado_elegido}: {V[estado_elegido]:.4f}")
print(f"Differenza: {abs(valor_manual - V[estado_elegido]):.6f}")

# Output atteso approssimativo:
# Valore calcolato a mano per (2, 3): -5.7568
# Valore della funzione generica per (2, 3): -5.7568
# Differenza: 0.0000

assert abs(valor_manual - V[estado_elegido]) < 1e-3, "Dovrebbero coincidere, entro la soglia di convergenza"
print("\nCoincidono: l'equazione di Bellman dell'aspettativa è soddisfatta nel punto fisso Vpi.")
```

**Perché funziona:** `evaluacion_iterativa_politica` non fa altro che applicare questo stesso calcolo a tutti i sedici stati, più volte, finché i valori smettono di muoversi. Quando ha già convergito, applicare l'equazione un'altra volta —a mano, per un solo stato— deve restituire (quasi) lo stesso numero già salvato in `V`. Questa è la definizione di punto fisso: uno stato in cui applicare l'equazione di Bellman non cambia più nulla.
</details>

---

## Per continuare a esercitarti

Se ti è avanzato del tempo, torna al `GridWorld` originale e prova queste varianti del capitolo:

1. **Cambia il fattore di sconto.** Prova con γ = 0.5 e con γ = 0.99. Come cambiano i valori delle caselle lontane dall'obiettivo? Che effetto ha sul numero di passate necessarie per convergere?
2. **Combina entrambi gli esercizi.** Ripeti l'esercizio 2 sul `mundo_ejercicio` dell'esercizio 1, invece che sul `mundo` originale, e verifica che l'equazione di Bellman continui a essere soddisfatta

**Prossima tappa:** nella Lezione 4 riutilizzerai questo stesso `GridWorld` per andare oltre la valutazione di una politica fissa: la migliorerai fino a trovare la politica ottimale π* tramite la programmazione dinamica. Salva questo notebook, ti servirà!